In [2]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, count, sum as _sum, year
import requests
import pandas as pd


pg_url = "jdbc:postgresql://metastore-db:5432/hive_metastore?createDatabaseIfNotExist=false&sslmode=disable"
pg_user = "hive"
pg_pass = "hivepass"

print("Iniciando Spark: ")

spark = SparkSession.builder \
    .appName("SECOP_ETL") \
    .master("spark://spark-master:7077") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .config("spark.sql.warehouse.dir", "hdfs://namenode:9000/user/hive/warehouse") \
    .config("spark.hadoop.javax.jdo.option.ConnectionURL", pg_url) \
    .config("spark.hadoop.javax.jdo.option.ConnectionDriverName", "org.postgresql.Driver") \
    .config("spark.hadoop.javax.jdo.option.ConnectionUserName", pg_user) \
    .config("spark.hadoop.javax.jdo.option.ConnectionPassword", pg_pass) \
    .config("spark.hadoop.datanucleus.schema.autoCreateAll", "false") \
    .config("spark.hadoop.hive.metastore.schema.verification", "false") \
    .enableHiveSupport() \
    .getOrCreate()

print(f"✅ Spark Listo. Versión: {spark.version}")

Iniciando Spark: 


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/14 01:40:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark Listo. Versión: 3.5.0


In [3]:
print("🕵️‍♀️ Validando conexión al Metastore...")
try:
    spark.sql("CREATE DATABASE IF NOT EXISTS test_connection")
    spark.sql("SHOW DATABASES").show()
    spark.sql("DROP DATABASE test_connection")
    spark.sql("DROP DATABASE secop")
    print("🎉 CONEXIÓN EXITOSA: El Metastore responde y la configuración es correcta.")
except Exception as e:
    print(f"❌ ERROR CRÍTICO: {e}")

🕵️‍♀️ Validando conexión al Metastore...


25/12/14 01:40:15 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
25/12/14 01:40:15 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
25/12/14 01:40:16 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
25/12/14 01:40:16 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore UNKNOWN@10.89.0.7
25/12/14 01:40:16 WARN ObjectStore: Failed to get database test_connection, returning NoSuchObjectException
25/12/14 01:40:16 WARN ObjectStore: Failed to get database test_connection, returning NoSuchObjectException
25/12/14 01:40:16 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
25/12/14 01:40:16 WARN ObjectStore: Failed to get database test_connection, returning NoSuchObjectException


+---------------+
|      namespace|
+---------------+
|        default|
|test_connection|
+---------------+

❌ ERROR CRÍTICO: [SCHEMA_NOT_FOUND] The schema `secop` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a catalog, verify the current_schema() output, or qualify the name with the correct catalog.
To tolerate the error on drop use DROP SCHEMA IF EXISTS.


25/12/14 01:40:17 WARN TxnHandler: Cannot perform cleanup since metastore table does not exist
25/12/14 01:40:17 WARN ObjectStore: Failed to get database secop, returning NoSuchObjectException


In [4]:
# Configuración de la API
BASE_URL = "https://www.datos.gov.co/resource/jbjy-vk9h.json"
LIMIT = 5000       # Tamaño del lote
Total_LIMIT = 20000 # Total a descargar (puedes subirlo si quieres)
offset = 0

# Ruta temporal en HDFS donde aterrizan los datos crudos
landing_path = "hdfs://namenode:9000/user/jovyan/secop_landing_json"

print(f"⬇️ Iniciando descarga hacia HDFS: {landing_path}")

# Limpiamos la zona de aterrizaje si ya existía (para empezar de cero)
# Usamos el FileSystem de Hadoop a través de la JVM de Spark
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
fs.delete(spark._jvm.org.apache.hadoop.fs.Path(landing_path), True)

while offset < Total_LIMIT:
    url = f"{BASE_URL}?$limit={LIMIT}&$offset={offset}"
    try:
        r = requests.get(url, timeout=20)
        batch = r.json()
    except Exception as e:
        print(f"❌ Error descargando offset {offset}: {e}")
        break

    if not batch:
        break

    # Truco Pro: Convertimos a Pandas -> String para evitar errores de esquema inferido
    # y luego pasamos a Spark para escribir en disco inmediatamente.
    pdf_small = pd.DataFrame(batch).astype(str)
    df_batch = spark.createDataFrame(pdf_small)
    
    # Escribimos en modo "append" (agregar al final)
    df_batch.write.mode("append").json(landing_path)
    
    print(f"  - Lote offset={offset} guardado en HDFS. RAM liberada.")
    
    # Limpieza manual de variables Python para asegurar memoria libre
    del batch, pdf_small, df_batch
    offset += LIMIT

print("✅ Ingesta finalizada. Los datos están seguros en HDFS.")


⬇️ Iniciando descarga hacia HDFS: hdfs://namenode:9000/user/jovyan/secop_landing_json


25/12/14 01:40:24 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
25/12/14 01:40:24 WARN TaskSetManager: Stage 0 contains a task of very large size (1089 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

  - Lote offset=0 guardado en HDFS. RAM liberada.


25/12/14 01:40:32 WARN TaskSetManager: Stage 1 contains a task of very large size (1093 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

  - Lote offset=5000 guardado en HDFS. RAM liberada.


25/12/14 01:40:39 WARN TaskSetManager: Stage 2 contains a task of very large size (1085 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

  - Lote offset=10000 guardado en HDFS. RAM liberada.


25/12/14 01:40:44 WARN TaskSetManager: Stage 3 contains a task of very large size (1089 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

  - Lote offset=15000 guardado en HDFS. RAM liberada.
✅ Ingesta finalizada. Los datos están seguros en HDFS.


In [ ]:
# --- EN TU NOTEBOOK DE EXTRACCIÓN (Extraccion.ipynb) ---

# Reutilizamos la configuración de Spark que ya funciona
# ... (Bloque de SparkSession) ...

import requests
import pandas
import time # Para usar 'sleep' en caso de error

BASE_URL = "https://www.datos.gov.co/resource/jbjy-vk9h.json"
LIMIT = 5000 # Un tamaño de lote seguro
offset = 0   # Contador de desplazamiento

print("Iniciando extracción por Paginación Infinita. ¡Ignorando el conteo!")

while True:
    try:
        # 1. Construir la URL con el desplazamiento actual
        url = f"{BASE_URL}?$limit={LIMIT}&$offset={offset}"
        
        # 2. Hacer la solicitud HTTP
        response = requests.get(url, timeout=120) # Aumentar el timeout por seguridad
        response.raise_for_status() # Lanza un error si hay fallos HTTP (4xx o 5xx)
        data = response.json()
        
        # 3. CRITERIO DE PARADA: Si la lista de datos está vacía, terminamos
        if not data:
            print(f"✅ ¡Extracción completa! El último lote con offset {offset} estaba vacío.")
            break 

        # 4. Procesar y Guardar el Lote
        pdf = pandas.DataFrame(data).astype(str)
        sdf = spark.createDataFrame(pdf)
        
        # Guardar en HDFS (Zona Bronce)
        # IMPORTANTE: Usar 'append' para ir añadiendo los lotes
        sdf.write.mode("append").json("hdfs://namenode:9000/datalake/bronze/secop_raw")
        
        print(f"⬇️ Lote {offset} guardado exitosamente. Registros: {len(data)}")
        
        # 5. Incrementar el desplazamiento para la próxima iteración
        offset += LIMIT

    except requests.exceptions.RequestException as e:
        print(f"❌ Error en la solicitud HTTP (Timeout o Red): {e}")
        print("Esperando 30 segundos antes de reintentar el mismo offset.")
        time.sleep(30)
        # NO aumentamos el offset aquí para reintentar el lote fallido

Iniciando extracción por Paginación Infinita. ¡Ignorando el conteo!


25/12/14 01:48:22 WARN TaskSetManager: Stage 4 contains a task of very large size (1089 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

⬇️ Lote 0 guardado exitosamente. Registros: 5000


25/12/14 01:48:29 WARN TaskSetManager: Stage 5 contains a task of very large size (1093 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

⬇️ Lote 5000 guardado exitosamente. Registros: 5000


...

⬇️ Lote 4505000 guardado exitosamente. Registros: 5000


25/12/14 03:40:36 WARN TaskSetManager: Stage 906 contains a task of very large size (1084 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

⬇️ Lote 4510000 guardado exitosamente. Registros: 5000


25/12/14 03:40:46 WARN TaskSetManager: Stage 907 contains a task of very large size (1087 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

⬇️ Lote 4515000 guardado exitosamente. Registros: 5000
